In [20]:
import pandas as pd 
import numpy as np

import os

# Stats

## Colunas numéricas

In [21]:
df_stats = pd.read_parquet('../data/raw/fighter_stats.parquet')
df_stats

,name,Height,Weight,Reach,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.
0,Charles Oliveira,"5' 10""",155 lbs.,"74""",Orthodox,"Oct 17, 1989",3.35,54%,3.24,49%,2.22,40%,54%,2.6
1,Max Holloway,"5' 11""",155 lbs.,"69""",Orthodox,"Dec 04, 1991",7.20,48%,4.74,59%,0.24,53%,83%,0.3


In [22]:
df_stats.columns

Index(['name', 'Height', 'Weight', 'Reach', 'STANCE', 'DOB', 'SLpM',
       'Str. Acc.', 'SApM', 'Str. Def', 'TD Avg.', 'TD Acc.', 'TD Def.',
       'Sub. Avg.'],
      dtype='object')

In [23]:
percent_cols = ['Str. Acc.', 'Str. Def', 'TD Acc.', 'TD Def.']

for col in percent_cols:
    df_stats[col] = df_stats[col].str.replace('%', '', regex=False)
    df_stats[col] = df_stats[col].astype(float) / 100.0

In [24]:
df_stats[['name'] + percent_cols]

,name,Str. Acc.,Str. Def,TD Acc.,TD Def.
0,Charles Oliveira,0.54,0.49,0.40,0.54
1,Max Holloway,0.48,0.59,0.53,0.83


## Altura e Envergadura padrão de gente normal (cm)

In [25]:
altura_certa = df_stats['Height'].str.replace('"', '', regex=False)

feet = altura_certa.str.split("'").str[0].astype(float)
inches = altura_certa.str.split("'").str[1].astype(float)

df_stats['Height'] = (feet * 30.48) + (inches * 2.54)

In [26]:
df_stats['Reach'] = df_stats['Reach'].str.replace('"', '', regex=False).astype(float)

df_stats['Reach_cm'] = df_stats['Reach'] * 2.54

## Peso 

In [27]:
df_stats['Weight'] = df_stats['Weight'].str.replace('lbs.', '', regex=False).astype(float)

df_stats.rename(columns={
    'Weight': 'Weight_lbs'}
    , inplace=True
    )

In [28]:
df_stats

,name,Height,Weight_lbs,Reach,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,Reach_cm
0,Charles Oliveira,177.80,155.0,74.0,Orthodox,"Oct 17, 1989",3.35,0.54,3.24,0.49,2.22,0.40,0.54,2.6,187.96
1,Max Holloway,180.34,155.0,69.0,Orthodox,"Dec 04, 1991",7.20,0.48,4.74,0.59,0.24,0.53,0.83,0.3,175.26


In [29]:
df_stats.drop('Reach', axis=1)

,name,Height,Weight_lbs,STANCE,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,Reach_cm
0,Charles Oliveira,177.80,155.0,Orthodox,"Oct 17, 1989",3.35,0.54,3.24,0.49,2.22,0.40,0.54,2.6,187.96
1,Max Holloway,180.34,155.0,Orthodox,"Dec 04, 1991",7.20,0.48,4.74,0.59,0.24,0.53,0.83,0.3,175.26


## Stance e salvar nova base

In [31]:
stance_map = {'Orthodox': 0, 'Southpaw': 1, 'Switch': 2}
df_stats['Stance_Code'] = df_stats['STANCE'].map(stance_map)

colunas_para_dropar = ['Height', 'STANCE']
df_stats_clean = df_stats.drop(columns=colunas_para_dropar)
df_stats.drop(columns=colunas_para_dropar)

,name,Weight_lbs,Reach,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,Reach_cm,Stance_Code
0,Charles Oliveira,155.0,74.0,"Oct 17, 1989",3.35,0.54,3.24,0.49,2.22,0.40,0.54,2.6,187.96,0
1,Max Holloway,155.0,69.0,"Dec 04, 1991",7.20,0.48,4.74,0.59,0.24,0.53,0.83,0.3,175.26,0


In [32]:
df_stats_clean.to_parquet('../data/processed/fighter_stats_clean.parquet', index=False)

# Fights

In [33]:
df_fights = pd.read_parquet('../data/raw/fight_history.parquet')

In [41]:
df_fights.head()

,Fighter,Opponent,Result,Method,Round,Time,Result_Code,Time_Seconds
0,Charles Oliveira,Mateusz Gamrot,win,SUB,2,2:48,1,168.0
1,Charles Oliveira,Ilia Topuria,loss,KO/TKO,1,2:27,0,147.0
2,Charles Oliveira,Michael Chandler,win,U-DEC,5,5:00,1,300.0
3,Charles Oliveira,Arman Tsarukyan,loss,S-DEC,3,5:00,0,300.0
4,Charles Oliveira,Beneil Dariush,win,KO/TKO,1,4:10,1,250.0


In [38]:
df_fights.dtypes

Fighter          object
Opponent         object
Result           object
Method           object
Round            object
Time             object
Result_Code       int64
Time_Seconds    float64
dtype: object

In [ ]:
## 1 -> vitoria
## 0 -> derrota
df_fights['Result_Code'] = (df_fights['Result'] == 'win').astype(int)

In [37]:
time_split = df_fights['Time'].str.split(':', expand=True)
minutos = time_split[0].astype(float)
segundos = time_split[1].astype(float)
df_fights['Time_Seconds'] = (minutos * 60) + segundos

In [39]:
df_fights['Round'] = df_fights['Round'].astype(int)

In [40]:
display(df_fights[['Fighter', 'Opponent', 'Result_Code', 'Round', 'Time_Seconds']].head(10))

,Fighter,Opponent,Result_Code,Round,Time_Seconds
0,Charles Oliveira,Mateusz Gamrot,1,2,168.0
1,Charles Oliveira,Ilia Topuria,0,1,147.0
2,Charles Oliveira,Michael Chandler,1,5,300.0
3,Charles Oliveira,Arman Tsarukyan,0,3,300.0
4,Charles Oliveira,Beneil Dariush,1,1,250.0
5,Charles Oliveira,Islam Makhachev,0,2,196.0
6,Charles Oliveira,Justin Gaethje,1,1,202.0
7,Charles Oliveira,Dustin Poirier,1,3,62.0
8,Charles Oliveira,Michael Chandler,1,2,19.0
9,Charles Oliveira,Tony Ferguson,1,3,300.0


## Salvar a base

In [42]:
colunas_para_dropar_historico = ['Result']
df_fights_clean = df_fights.drop(columns=colunas_para_dropar_historico)

In [44]:
df_final = pd.merge(df_fights_clean, df_stats_clean, how='left', left_on='Fighter', right_on='name')

In [48]:
df_final = df_final.drop(columns=['name'])

In [53]:
display(df_final.head())

,Fighter,Opponent,Method,Round,Time,Result_Code,Time_Seconds,Weight_lbs,Reach,DOB,SLpM,Str. Acc.,SApM,Str. Def,TD Avg.,TD Acc.,TD Def.,Sub. Avg.,Reach_cm,Stance_Code
0,Charles Oliveira,Mateusz Gamrot,SUB,2,2:48,1,168.0,155.0,74.0,"Oct 17, 1989",3.35,0.54,3.24,0.49,2.22,0.4,0.54,2.6,187.96,0
1,Charles Oliveira,Ilia Topuria,KO/TKO,1,2:27,0,147.0,155.0,74.0,"Oct 17, 1989",3.35,0.54,3.24,0.49,2.22,0.4,0.54,2.6,187.96,0
2,Charles Oliveira,Michael Chandler,U-DEC,5,5:00,1,300.0,155.0,74.0,"Oct 17, 1989",3.35,0.54,3.24,0.49,2.22,0.4,0.54,2.6,187.96,0
3,Charles Oliveira,Arman Tsarukyan,S-DEC,3,5:00,0,300.0,155.0,74.0,"Oct 17, 1989",3.35,0.54,3.24,0.49,2.22,0.4,0.54,2.6,187.96,0
4,Charles Oliveira,Beneil Dariush,KO/TKO,1,4:10,1,250.0,155.0,74.0,"Oct 17, 1989",3.35,0.54,3.24,0.49,2.22,0.4,0.54,2.6,187.96,0
